## Taxi Data

### Setup   

In [2]:
# libraries
import matplotlib.pyplot as plt ## for basic plotting
import matplotlib as mpl ## for setting default parameters
import pandas as pd ## always
import os ## for handling file paths 1
from pathlib import Path ## for handling file paths 2
import numpy as np ## for numerical operations
import seaborn as sns ## for more advanced plotting

In [3]:
# paths
base_dir = Path("/Users/hannahmaihojgaard/Documents/GitHub/datascience2026")
data_path = Path(base_dir / "data")

### Load in data

In [3]:
# find all parquet files
files = sorted(data_path.glob("yellow_tripdata_*.parquet"))

# check files
print(f"Found {len(files)} files")

Found 29 files


In [4]:
# create empty list
dfs = []

# loop through files
for file in files:
    print(f"Loading {file.name}")

    df = pd.read_parquet(file)

    # optional: add source year/month
    year_month = file.stem.replace("yellow_tripdata_", "")
    df["source_file"] = year_month

    dfs.append(df)

# combine into one dataframe
taxi_df = pd.concat(dfs, ignore_index=True)

print("Done!")
print(taxi_df.shape)

Loading yellow_tripdata_2023-07.parquet
Loading yellow_tripdata_2023-08.parquet
Loading yellow_tripdata_2023-09.parquet
Loading yellow_tripdata_2023-10.parquet
Loading yellow_tripdata_2023-11.parquet
Loading yellow_tripdata_2023-12.parquet
Loading yellow_tripdata_2024-01.parquet
Loading yellow_tripdata_2024-02.parquet
Loading yellow_tripdata_2024-03.parquet
Loading yellow_tripdata_2024-04.parquet
Loading yellow_tripdata_2024-05.parquet
Loading yellow_tripdata_2024-06.parquet
Loading yellow_tripdata_2024-08.parquet
Loading yellow_tripdata_2024-09.parquet
Loading yellow_tripdata_2024-10.parquet
Loading yellow_tripdata_2024-11.parquet
Loading yellow_tripdata_2024-12.parquet
Loading yellow_tripdata_2025-01.parquet
Loading yellow_tripdata_2025-02.parquet
Loading yellow_tripdata_2025-03.parquet
Loading yellow_tripdata_2025-04.parquet
Loading yellow_tripdata_2025-05.parquet
Loading yellow_tripdata_2025-06.parquet
Loading yellow_tripdata_2025-07.parquet
Loading yellow_tripdata_2025-08.parquet


### Preproccessing


- *VendorID*: Taxi technology provider (1 = CMT, 2 = VeriFone)
- *tpep_pickup_datetime*: Timestamp when the trip started (meter on)
- *tpep_dropoff_datetime*: Timestamp when the trip ended (meter off)
- *passenger_count*: Number of passengers (driver-reported)
- *trip_distance*: Trip distance in miles (from taximeter)
- *RatecodeID*: Rate type (e.g., standard, JFK, Newark, etc.)
- *store_and_fwd_flag*: Whether trip data was stored before sending (Y/N)
- *PULocationID*: Pickup location ID (zone-based)
- *DOLocationID*: Dropoff location ID (zone-based)
- *payment_type*: Payment method (1=card, 2=cash, etc.)
- *fare_amount*: Base fare (time + distance)
- *extra*: Additional charges (e.g., rush hour, overnight)
- *mta_tax*: Fixed $0.50 MTA tax
- *tip_amount*: Tip (only recorded for card payments)
- *tolls_amount*: Total toll charges
- *improvement_surcharge*: $0.30 improvement fee
- *total_amount*: Total charged (excluding cash tips)
- *congestion_surcharge*: Fee for trips in congestion zones
- *Airport_fee*: Extra fee for airport trips
- *cbd_congestion_fee*: Central business district congestion fee


In [5]:
taxi_df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,source_file,cbd_congestion_fee
0,1,2023-07-01 00:29:59,2023-07-01 00:40:15,1.0,1.80,1.0,N,140,263,1,...,3.5,0.5,5.10,0.0,1.0,22.20,2.5,0.0,2023-07,NaN
1,2,2023-07-01 00:03:25,2023-07-01 00:23:44,1.0,2.31,1.0,N,163,163,2,...,1.0,0.5,0.00,0.0,1.0,24.10,2.5,0.0,2023-07,NaN
2,2,2023-07-01 00:38:29,2023-07-01 00:48:53,1.0,2.36,1.0,N,142,262,1,...,1.0,0.5,3.70,0.0,1.0,22.20,2.5,0.0,2023-07,NaN
3,2,2023-07-01 00:14:16,2023-07-01 00:29:13,1.0,4.36,1.0,N,68,24,1,...,1.0,0.5,4.96,0.0,1.0,29.76,2.5,0.0,2023-07,NaN
4,1,2023-07-01 00:11:15,2023-07-01 00:20:47,0.0,1.60,1.0,N,161,107,1,...,3.5,0.5,3.25,0.0,1.0,19.65,2.5,0.0,2023-07,NaN


### Removing NaN's

In [6]:
#Imputing NaN values in fee columns with 0

fee_columns = [
    "Airport_fee",
    "cbd_congestion_fee",
    "congestion_surcharge",
    "tolls_amount",
    "extra",
    "mta_tax"
]

taxi_df[fee_columns] = taxi_df[fee_columns].fillna(0)

In [7]:
# check for remaining NaN values in tip_amount
taxi_df["tip_amount"].isna().sum()

0

### Trip Duration

In [8]:
# calculate trip duration in minutes
taxi_df["trip_duration"] = (
    taxi_df["tpep_dropoff_datetime"] - taxi_df["tpep_pickup_datetime"]
).dt.total_seconds() / 60

In [9]:
# identify trips with invalid durations (<= 0 or > 180 minutes) - these are likely data errors or outliers
invalid_duration = taxi_df[
    (taxi_df["trip_duration"] <= 0) |
    (taxi_df["trip_duration"] > 180)
]

invalid_duration.shape

(624555, 22)

In [10]:
# filter out trips with invalid durations
taxi_df = taxi_df[
    (taxi_df["trip_duration"] > 0) &
    (taxi_df["trip_duration"] <= 180)
]

In [11]:
# check trip duration statistics
taxi_df["trip_duration"].describe()

count    1.050075e+08
mean     1.698982e+01
std      1.375232e+01
min      1.666667e-02
25%      8.033333e+00
50%      1.330000e+01
75%      2.141667e+01
max      1.800000e+02
Name: trip_duration, dtype: float64

In [12]:
# check shape after cleaning
taxi_df.shape

(105007470, 22)

### Trip Distance

In [13]:
# check trip distance statistics
taxi_df["trip_distance"].describe()

count    1.050075e+08
mean     5.688879e+00
std      5.187476e+02
min      0.000000e+00
25%      1.020000e+00
50%      1.800000e+00
75%      3.510000e+00
max      3.986086e+05
Name: trip_distance, dtype: float64

In [14]:
# check for trips with non-positive distance
taxi_df[taxi_df["trip_distance"] <= 0].shape

(2600927, 22)

In [15]:
# filter out trips with non-positive distance
taxi_df = taxi_df[
    taxi_df["trip_distance"] > 0
]

In [16]:
# check trip distance statistics after filtering
taxi_df["trip_distance"].describe()

count    1.024065e+08
mean     5.833366e+00
std      5.252931e+02
min      1.000000e-02
25%      1.090000e+00
50%      1.840000e+00
75%      3.600000e+00
max      3.986086e+05
Name: trip_distance, dtype: float64

In [17]:
# check for trips with extremely long distances (e.g. > 100 miles) - these are likely data errors or outliers
taxi_df[taxi_df["trip_distance"] > 100].shape
taxi_df = taxi_df[
    taxi_df["trip_distance"] <= 100
]

In [18]:
taxi_df["trip_distance"].describe()

count    1.024022e+08
mean     3.479821e+00
std      4.406514e+00
min      1.000000e-02
25%      1.090000e+00
50%      1.840000e+00
75%      3.600000e+00
max      9.991000e+01
Name: trip_distance, dtype: float64

### Passenger Count

In [19]:
taxi_df["passenger_count"].describe()

# check distribution of passenger counts
taxi_df["passenger_count"].value_counts().sort_index()

passenger_count
0.0      853771
1.0    68054136
2.0    12668912
3.0     3012911
4.0     1910340
5.0      688228
6.0      445513
7.0          75
8.0         213
9.0          53
Name: count, dtype: int64

In [20]:
# filter out trips with unrealistic passenger counts (e.g. > 6) - these are likely data errors
taxi_df = taxi_df[
    taxi_df["passenger_count"] <= 6
]

# check distribution of passenger counts again after filtering
taxi_df["passenger_count"].value_counts().sort_index()

passenger_count
0.0      853771
1.0    68054136
2.0    12668912
3.0     3012911
4.0     1910340
5.0      688228
6.0      445513
Name: count, dtype: int64

### Fare Amount

In [21]:
# check fare amount statistics
taxi_df["fare_amount"].describe()



count    8.763381e+07
mean     1.910119e+01
std      1.157543e+02
min     -1.508700e+03
25%      9.300000e+00
50%      1.350000e+01
75%      2.190000e+01
max      8.633721e+05
Name: fare_amount, dtype: float64

In [22]:
# check for trips with zero or negative fare amounts - these are likely data errors
taxi_df[taxi_df["fare_amount"] <= 0].shape

(1459363, 22)

In [23]:
# filter out trips with zero or negative fare amounts
taxi_df = taxi_df[
    taxi_df["fare_amount"] > 0
]

# check fare amount statistics again after filtering
taxi_df["fare_amount"].describe()

count    8.617445e+07
mean     1.980902e+01
std      1.165527e+02
min      1.000000e-02
25%      9.300000e+00
50%      1.350000e+01
75%      2.190000e+01
max      8.633721e+05
Name: fare_amount, dtype: float64

In [24]:
taxi_df = taxi_df[
    taxi_df["fare_amount"] <= 500
]

In [25]:
taxi_df["fare_amount"].describe()

count    8.617390e+07
mean     1.977845e+01
std      1.818251e+01
min      1.000000e-02
25%      9.300000e+00
50%      1.350000e+01
75%      2.190000e+01
max      5.000000e+02
Name: fare_amount, dtype: float64

### Save df

In [26]:
# save cleaned dataframe
taxi_df.to_parquet(
    data_path / "taxi_cleaned_part2.parquet",
    index=False
)

## Fix date errors

In [4]:
# load cleaned taxi datasets
taxi_df_1 = pd.read_parquet(
    data_path / "taxi_cleaned_part1.parquet"
)

taxi_df_2 = pd.read_parquet(
    data_path / "taxi_cleaned_part2.parquet"
)

In [5]:
taxi_df_1 = taxi_df_1[
    (taxi_df_1["tpep_pickup_datetime"].dt.year >= 2021) &
    (taxi_df_1["tpep_pickup_datetime"].dt.year <= 2025)
]


taxi_df_2 = taxi_df_2[
    (taxi_df_2["tpep_pickup_datetime"].dt.year >= 2021) &
    (taxi_df_2["tpep_pickup_datetime"].dt.year <= 2025)
]   

In [6]:
# save cleaned dataframe
taxi_df_1.to_parquet(
    data_path / "taxi_cleaned_part1.parquet",
    index=False
)

taxi_df_2.to_parquet(
    data_path / "taxi_cleaned_part2.parquet",
    index=False
)